In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import matplotlib.pyplot as plt
import os

df_path = os.path.join(path, 'Q1_data.csv')

df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.preprocessing import StandardScaler

df.drop(columns=['Order_ID'], inplace=True)
df.head()

In [ ]:
# Task 2: Write your code here:

# step1: check the missing values:
print("Missing values:")
print(df.isnull().sum())

# step2: handling:
# step2.1: we separate numerical and categorical to fill the numerical missing values with the median and the categorical missing values with the mode

num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# step3: re-check
print('-----------------------')
print("Missing values after handling:")
print(df.isnull().sum())


In [ ]:
# Task 3: Write your code here:

print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

# Apply One-Hot Encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Check result
df.head()


In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_scaled.head()


In [ ]:
# Task 6: Write your code here:
print("Features shape:", X_scaled.shape)
print("Target shape:", y.shape)

In [ ]:
# Task 1: Write your code here:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X = X_scaled
y = y

In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
#----
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)
  #----
print("MAE scores per fold:", mae_scores)
print("Average MAE:", np.mean(mae_scores))

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt

# Train model on full dataset for feature importance
final_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
final_model.fit(X, y)

importances = final_model.feature_importances_
indices = np.argsort(importances)[-10:]  # top 10 features

plt.figure()
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), X.columns[indices])
plt.xlabel("Feature Importance")
plt.title("Top 10 Feature Importances")
plt.show()


In [ ]:
# Task 2: Write your code here:
# Predict delivery time
y_pred_all = final_model.predict(X)

plt.figure()
plt.hist(y_pred_all, bins=30)
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Predicted Delivery Time Distribution")
plt.show()


In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostRegressor
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores_ensemble = []

for train_index, test_index in kf.split(X_scaled):
    X_train, X_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    cb_model = CatBoostRegressor(
        iterations=500, learning_rate=0.1, depth=6, verbose=0, random_state=42
    )
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_test)

    avg_pred = (rf_pred + cb_pred) / 2

    mae = mean_absolute_error(y_test, avg_pred)
    mae_scores_ensemble.append(mae)
# results:
print("Ensemble MAE per fold:", mae_scores_ensemble)
print("Average Ensemble MAE:", np.mean(mae_scores_ensemble))